In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, log_loss, accuracy_score
from sklearn.preprocessing import  OneHotEncoder, LabelEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from tqdm import tqdm

In [6]:
sonar = pd.read_csv("Sonar.csv")
le = LabelEncoder()
y = le.fit_transform( sonar['Class'] )
X = sonar.drop('Class', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,
                                                    stratify=y)

In [7]:
dtc = DecisionTreeClassifier(random_state=25, max_depth=1)
nb = GaussianNB()
lr = LogisticRegression()

In [8]:
ada = AdaBoostClassifier(estimator=dtc, n_estimators=10, random_state=25)
ada.fit(X_train, y_train)
y_pred = ada.predict(X_test)
print( classification_report(y_test, y_pred) )
y_pred_prob = ada.predict_proba(X_test)
print( log_loss(y_test, y_pred_prob) )

              precision    recall  f1-score   support

           0       0.73      0.88      0.80        34
           1       0.82      0.62      0.71        29

    accuracy                           0.76        63
   macro avg       0.77      0.75      0.75        63
weighted avg       0.77      0.76      0.76        63

0.5399269544547101


In [10]:
ests = [dtc, nb, lr]
n_ests = [10, 25, 35, 60]
scores = []
for e in tqdm( ests ):
    for n in n_ests:
        ada = AdaBoostClassifier(estimator=e, n_estimators=n, random_state=25)
        ada.fit(X_train, y_train)
        y_pred_prob = ada.predict_proba(X_test)
        scores.append([e,n, log_loss(y_test, y_pred_prob) ])
df_scores = pd.DataFrame( scores, columns=['estimator','n_estimators','score'] )
df_scores.sort_values( 'score',ascending=True )

100%|██████████| 3/3 [00:01<00:00,  2.67it/s]


,estimator,n_estimators,score
6,GaussianNB(),35,0.499461
7,GaussianNB(),60,0.502435
5,GaussianNB(),25,0.506931
4,GaussianNB(),10,0.520720
10,LogisticRegression(),35,0.537005
11,LogisticRegression(),60,0.537005
0,"DecisionTreeClassifier(max_depth=1, random_sta...",10,0.539927
9,LogisticRegression(),25,0.540850
8,LogisticRegression(),10,0.544556
1,"DecisionTreeClassifier(max_depth=1, random_sta...",25,0.561139
